# CareLoom Fundamentals — LangGraph Basics at Larkspur Electronics

**Larkspur Electronics** sells keyboards, mice, headphones, monitors, and other desk gear
direct to consumers across India. Every inbound message — *"where's my order"*, *"does this
keyboard work with a Mac"*, *"can I return this"* — currently lands in the same support
queue and gets triaged by hand before a human even reads it.

**CareLoom** is the agentic system Larkspur's platform team is building to change that. This
notebook is where it starts: the smallest possible LangGraph graph that reads an incoming
message, decides whether it's an *order* question or a *product* question, and drafts a
first-pass reply. Nothing here calls an LLM yet — the goal is to get comfortable with the
graph mechanics themselves before wiring in Groq in the next notebook.

This notebook introduces the core ideas behind LangGraph:

- a typed `StateGraph`
- nodes and conditional edges
- compiling and invoking a graph
- graph visualization
- LangSmith node spans with `@traceable`

LangGraph models workflows as graphs made of state, nodes, and edges. The docs say the main graph class is `StateGraph`, the state schema is commonly defined with `TypedDict` or a Pydantic model, and graphs must be compiled before use.

## Learning goals

By the end of this notebook, you should be able to:

1. Define graph state with a `TypedDict`.
2. Add nodes to a `StateGraph`.
3. Route with conditional edges.
4. Compile and invoke the graph.
5. Visualize the graph.
6. Trace node spans in LangSmith.


## 1) Install packages

```bash
pip install -U langgraph langsmith
```

LangSmith's annotate-code guide says `LANGSMITH_TRACING` controls `@traceable`, and `LANGSMITH_PROJECT` sets the project name.


In [ ]:
%pip install -qU langgraph langsmith


## 2) Configure LangSmith

Set these in your environment or `.env` file:

```env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=your_key
LANGSMITH_PROJECT=careloom-fundamentals
```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_API_KEY set:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT", "careloom-fundamentals"))


## 3) Import LangGraph pieces


In [ ]:
from typing_extensions import TypedDict
from typing import Literal

from langgraph.graph import StateGraph, START, END
from langsmith import traceable


## 4) Define graph state

The LangGraph docs say graph state is commonly defined with `TypedDict`. For this simple lab we only need a few keys and no custom reducer.


In [ ]:
class GraphState(TypedDict):
    message: str
    route: str
    draft: str
    final: str


## 5) Define the nodes

We will build a small workflow:

- `classify_message`: decide whether an inbound message is an order question or a product question
- `order_help_node`: draft a reply for order/shipping/refund questions
- `product_help_node`: draft a reply for product/spec questions
- `finalize_node`: assemble the final answer

The docs describe nodes as functions that receive state, do work, and return updated state.


In [ ]:
ORDER_WORDS = ["order", "shipping", "delivery", "deliver", "track", "dispatch", "refund", "return", "cancel"]


@traceable(name="classify_message", run_type="chain")
def classify_message(state: GraphState) -> dict:
    message = state["message"].lower()
    if any(word in message for word in ORDER_WORDS):
        route = "order_help"
    else:
        route = "product_help"
    return {"route": route}

@traceable(name="order_help_node", run_type="chain")
def order_help_node(state: GraphState) -> dict:
    message = state["message"]
    return {"draft": f'Thanks for reaching out about your order. Pulling up tracking/refund details for: "{message}"'}

@traceable(name="product_help_node", run_type="chain")
def product_help_node(state: GraphState) -> dict:
    message = state["message"]
    return {"draft": f'Happy to help with product info. Checking specs and compatibility for: "{message}"'}

@traceable(name="finalize_node", run_type="chain")
def finalize_node(state: GraphState) -> dict:
    return {"final": f"Message: {state['message']} | Route: {state['route']} | Draft reply: {state['draft']}"}


## 6) Conditional edges

The LangGraph docs say conditional edges call a routing function after a node runs and then choose the next node based on the returned value. They also note that you should use either static edges or dynamic routing from the same node, not both.


In [ ]:
def route_after_classify(state: GraphState) -> Literal["order_help", "product_help"]:
    return state["route"]

builder = StateGraph(GraphState)

builder.add_node("classify_message", classify_message)
builder.add_node("order_help_node", order_help_node)
builder.add_node("product_help_node", product_help_node)
builder.add_node("finalize_node", finalize_node)

builder.add_edge(START, "classify_message")
builder.add_conditional_edges(
    "classify_message",
    route_after_classify,
    {"order_help": "order_help_node", "product_help": "product_help_node"},
)
builder.add_edge("order_help_node", "finalize_node")
builder.add_edge("product_help_node", "finalize_node")
builder.add_edge("finalize_node", END)

graph = builder.compile()

print("Graph compiled successfully.")


## 7) Invoke the graph

Graphs must be compiled before use. The docs explicitly say you must call `.compile()` before invoking a graph.


In [ ]:
result = graph.invoke({
    "message": "Where is my order? It hasn't arrived yet.",
    "route": "",
    "draft": "",
    "final": "",
})

result


In [ ]:
result_2 = graph.invoke({
    "message": "Does the Aria wireless keyboard work with a Mac?",
    "route": "",
    "draft": "",
    "final": "",
})

result_2


## 8) Visualize the graph

The use-graph-api docs show that you can visualize a graph and convert it to Mermaid syntax with `app.get_graph().draw_mermaid()`.


In [ ]:
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)


## 9) LangSmith node spans

LangSmith's annotate-code docs say `@traceable` turns a function into a traced run, and nested traced functions become child spans automatically. That means each node in this notebook appears as its own span in LangSmith when tracing is enabled.


In [ ]:
demo = graph.invoke({
    "message": "Can I return my headphones if I already opened the box?",
    "route": "",
    "draft": "",
    "final": "",
})

demo


## 10) A small mental model

Use this mapping:

- `State` = the shared notebook of the graph — here, one inbound Larkspur message as it moves through triage
- `Node` = one step that reads state and writes updates
- `Conditional edge` = a routing decision (order question vs. product question)
- `Compile` = validate and prepare the graph
- `Invoke` = run the graph once, for one message
- `Traceable node` = a span in LangSmith

## 11) Common mistakes

1. Forgetting to compile the graph.
2. Mixing static edges and dynamic routing from the same node.
3. Returning the wrong keys from a node.
4. Using unclear state.
5. Forgetting to turn on LangSmith tracing when you want spans.

The LangGraph docs warn against mixing static edges with dynamic routing from the same node because both paths can run and make the graph harder to reason about.

## Key takeaways

- `StateGraph` is the main graph builder.
- `TypedDict` is the beginner-friendly way to define state.
- Conditional edges route the graph based on a function output — here, a simple keyword check that stands in for the intent classifier CareLoom will eventually use.
- Graphs must be compiled before invocation.
- `draw_mermaid()` helps you visualize the graph.
- `@traceable` gives you LangSmith node spans and nested tracing.

This tiny triage graph is intentionally low-stakes: no LLM, no external services, nothing that can fail from a missing API key. The next notebook keeps the same shape — classify, route, respond — and swaps in a real Groq-backed agent with tools.

## References

- Graph API overview: https://docs.langchain.com/oss/python/langgraph/graph-api
- Conditional edges: https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges
- Visualize your graph: https://docs.langchain.com/oss/python/langgraph/use-graph-api#visualize-your-graph
- Annotate code: https://docs.langchain.com/langsmith/annotate-code
